# Teste avaliativo para vaga de bolsista em engenharia/análise de dados

## 1. Extração dos dados

In [65]:
import requests
import time
import json
import pandas as pd
import pickle
import os

In [ ]:
# CONFIGURAR PASTA PARA SALVAR OS DADOS DA REQUISIÇÃO
os.makedirs('data', exist_ok=True)


URL = 'https://api.obrasgov.gestao.gov.br/obrasgov/api'
ENDPOINT = 'projeto-investimento'
MAX_ATTEMPT = 2
PAGE_SIZE = 10
UF = 'DF'
page = 0 


# PROJETOS DO DF
projetos_DF = []

# BUSCAR DADOS UTILIZANDO A API FORNECIDA
while True:
    response = requests.get(f'{URL}/{ENDPOINT}?uf={UF}&pagina={page}&tamanhoDaPagina={PAGE_SIZE}')

    # se vir um erro 404 indica que aquela página não existe ou a requisição está falhando
    if response.status_code == 404:
        print(f'Página {page} não existe. Finalizando.')
        break

    # se vir um erro 429, siginifica que meu ip tá sendo negado, preciso aguardar o rate-limit imposto pela api
    elif response.status_code == 429:
        retry_after = int(response.headers.get('x-rate-limit-retry-after-seconds', 2))
        print(f'Muitas requisições! Aguardando {retry_after} segundos...')
        time.sleep(retry_after)
        continue

    try:
        data = response.json()
    except ValueError as e:
        print(f'Erro ao decodificar JSON na página {page}: {e}')
        break

    # se pagina estiver vazia:
    if not data.get("content") or data.get("empty", False):
        print(f'Acabaram os dados para o UF = {UF}. ({page=})')
        break

    for item in data.get('content', []):
        projetos_DF.append(item)

    page += 1
    time.sleep(2)
        

print(f'Dados carregados. Total de dados encontrados = {len(projetos_DF)}.')


# SALVAR DADOS EM JSON:
json_path = os.path.join('data', 'projetos_DF.json')
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(projetos_DF, file, ensure_ascii=False, indent=2)
print(f'Arquivo JSON salvo: {json_path}')


# SALVAR DADOS EM CSV:
csv_path = os.path.join('data', 'projetos_DF.csv')
df = pd.DataFrame(projetos_DF)
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f'Arquivo CSV salvo: {csv_path}')


# SALVAR DADOS EM PICKLE:
pkl_path = os.path.join('data', 'projetos_DF.pkl')
with open(pkl_path, "wb") as file:
    pickle.dump(projetos_DF, file)
print(f'Arquivo Pickle salvo: {pkl_path}')

Acabaram os dados para o UF = DF. (page=84)
Dados carregados. Total de dados encontrados = 834.
Arquivo JSON salvo: data/projetos_DF.json
Arquivo CSV salvo: data/projetos_DF.csv
Arquivo Pickle salvo: data/projetos_DF.pkl


## 2. Busca exploratória dos dados

- Visão geral dos dados: colunas, dimensões, tipos de variáveis, taxas de valores nulos.
- Estatísticas descritivas
- Qualidade dos dados​

### 2.1 Visão geral dos dados

In [145]:
# CARREGAR OS DADOS VINDOS DE ALGUM ARQUIVO SALVO ACIMA, PARA ESSE EXEMPLO, IREI USAR OS DADOS VINDOS DO CSV:
df = pd.read_csv('data/projetos_DF.csv')

# quantidade de linhas e colunas:
rows, cols = df.shape

print(f'Dimensões (Linhas x Colunas): {rows} x {cols}')
print(f'Quantidade de linhas: {rows}')
print(f'Quantidade de colunas: {cols}')
print(f'Total de dados: (linhas x colunas) = {rows * cols}\n')

Dimensões (Linhas x Colunas): 834 x 31
Quantidade de linhas: 834
Quantidade de colunas: 31
Total de dados: (linhas x colunas) = 25854



In [137]:
# quantidade de valores nulos
qtd_null = df.isnull().sum()

# porcentagem de valores nulos:
percent_null = qtd_null / len(df) * 100

print("DADOS NULOS POR COLUNAS DOS DADOS OBTIDOS:\n")

for col in df.columns:
    print(f'{col}: {qtd_null[col]} --- ({percent_null[col]:.2f}%)')

DADOS NULOS POR COLUNAS DOS DADOS OBTIDOS:

idUnico: 0 --- (0.00%)
nome: 0 --- (0.00%)
cep: 453 --- (54.32%)
endereco: 445 --- (53.36%)
descricao: 0 --- (0.00%)
funcaoSocial: 0 --- (0.00%)
metaGlobal: 0 --- (0.00%)
dataInicialPrevista: 2 --- (0.24%)
dataFinalPrevista: 2 --- (0.24%)
dataInicialEfetiva: 812 --- (97.36%)
dataFinalEfetiva: 828 --- (99.28%)
dataCadastro: 0 --- (0.00%)
especie: 7 --- (0.84%)
natureza: 0 --- (0.00%)
naturezaOutras: 763 --- (91.49%)
situacao: 0 --- (0.00%)
descPlanoNacionalPoliticaVinculado: 576 --- (69.06%)
uf: 0 --- (0.00%)
qdtEmpregosGerados: 673 --- (80.70%)
descPopulacaoBeneficiada: 674 --- (80.82%)
populacaoBeneficiada: 678 --- (81.29%)
observacoesPertinentes: 718 --- (86.09%)
isModeladaPorBim: 259 --- (31.06%)
dataSituacao: 0 --- (0.00%)
tomadores: 0 --- (0.00%)
executores: 0 --- (0.00%)
repassadores: 0 --- (0.00%)
eixos: 0 --- (0.00%)
tipos: 0 --- (0.00%)
subTipos: 0 --- (0.00%)
fontesDeRecurso: 0 --- (0.00%)


In [138]:
# informações do dataframe:

print('INFORMAÇÕES DO DATAFRAME:\n')
df.info()

INFORMAÇÕES DO DATAFRAME:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 834 entries, 0 to 833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   idUnico                             834 non-null    object
 1   nome                                834 non-null    object
 2   cep                                 381 non-null    object
 3   endereco                            389 non-null    object
 4   descricao                           834 non-null    object
 5   funcaoSocial                        834 non-null    object
 6   metaGlobal                          834 non-null    object
 7   dataInicialPrevista                 832 non-null    object
 8   dataFinalPrevista                   832 non-null    object
 9   dataInicialEfetiva                  22 non-null     object
 10  dataFinalEfetiva                    6 non-null      object
 11  dataCadastro                   

### 2.2 Informações descritivas dos dados

In [139]:
# informações descritivas do dataframe:

print('INFORMAÇÕES DESCRITIVAS DO DATAFRAME:\n')
print('count = quantidade de valores não nulos')
print('unique = quantidade de valores únicos')
print('top = valor mais frequente')
print('freq = frequência do valor mais comum')
print('dtype = tipo do dado (será tratado mais para a frente)')

for col in df.columns:
    print(f"\n=== Estatísticas da coluna '{col}' ===\n")
    print(df[col].describe(), "\n")

INFORMAÇÕES DESCRITIVAS DO DATAFRAME:

count = quantidade de valores não nulos
unique = quantidade de valores únicos
top = valor mais frequente
freq = frequência do valor mais comum
dtype = tipo do dado (será tratado mais para a frente)

=== Estatísticas da coluna 'idUnico' ===

count             834
unique            606
top       60895.53-14
freq                4
Name: idUnico, dtype: object 


=== Estatísticas da coluna 'nome' ===

count                                       834
unique                                      570
top       CONSTRUÇÃO DE UNIDADE BÁSICA DE SAÚDE
freq                                          9
Name: nome, dtype: object 


=== Estatísticas da coluna 'cep' ===

count     381
unique     85
top         1
freq       92
Name: cep, dtype: object 


=== Estatísticas da coluna 'endereco' ===

count         389
unique        220
top        BR-010
freq           12
Name: endereco, dtype: object 


=== Estatísticas da coluna 'descricao' ===

count                     

### 2.3 Qualidade dos dados

In [146]:
# qualidade dos dados:

print('QUALIDADE DOS DADOS DO DATAFRAME:\n')

# quantidade de linhas duplicadas:
rows_duplicated = df.duplicated().sum()
print(f'Quantidade de linhas duplicadas: {rows_duplicated}\n')

# total de valores nulos (sendo que o total de valores foi dito acima)
total_null = df.isnull().sum().sum()
print(f'Total de valores nulos: {total_null}\n')

# exemplo de linhas com valores nulos:
rows_nulls = df[df.isnull().any(axis=1)]
print(f'Exemplo de linhas com valores nulos:')
rows_nulls.head()

QUALIDADE DOS DADOS DO DATAFRAME:

Quantidade de linhas duplicadas: 188

Total de valores nulos: 6890

Exemplo de linhas com valores nulos:


,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,NaN,NaN,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,NaN,...,NaN,False,2024-12-20,[],[{'nome': 'DEPARTAMENTO NACIONAL DE INFRAESTRU...,[],"[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 25, 'descricao': 'Rodovia', 'idEixo': 3}]","[{'id': 4, 'descricao': 'Acessos Terrestres', ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
1,42724.53-27,Escola Classe Crixá São Sebastião,NaN,NaN,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,NaN,...,NaN,False,2025-09-05,[],[{'nome': 'SECRETARIA DE ESTADO DE EDUCACAO DO...,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,"[{'id': 4, 'descricao': 'Social'}]","[{'id': 46, 'descricao': 'Educação', 'idEixo':...","[{'id': 84, 'descricao': 'Educação', 'idTipo':...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70.602-600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,NaN,...,NaN,False,2023-02-06,[],[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 1, 'descricao': 'Segurança Pública', '...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,NaN,NaN,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,NaN,...,NaN,False,2023-08-28,[],[{'nome': 'DEPARTAMENTO DE ESTRADAS DE RODAGEM...,"[{'nome': 'MINISTÉRIO DAS CIDADES', 'codigo': ...","[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 24, 'descricao': 'Infraestrutura Urban...","[{'id': 57, 'descricao': 'Obra de Arte Especia...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",NaN,NaN,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,NaN,...,NaN,False,2023-08-29,[],"[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'id': 3, 'descricao': 'Econômico'}, {'id': 3...","[{'id': 31, 'descricao': 'Energia', 'idEixo': ...","[{'id': 95, 'descricao': 'Subestação', 'idTipo...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
